# 56 | Evaluación final en test de los modelos con presupuesto equiparado (NB54 + NB55)

**Qué hace:** evalúa **una sola vez** en el conjunto de test (373 imágenes, 4.648 cerdos, 5 fechas) los 12 modelos entrenados con el mismo presupuesto de 60 min en L4:

| Configuración | Origen | Semillas |
|---|---|---|
| YOLO11n-seg | NB54 | 2026, 2027, 2028 |
| YOLO26n-seg | NB54 | 2026, 2027, 2028 |
| Mask R-CNN, LR constante | NB54 | 2026, 2027, 2028 |
| Mask R-CNN, LR coseno | NB55 | 2026, 2027, 2028 |

Usa **exactamente** el mismo protocolo COCO del NB33 (misma verdad de terreno, `COCOeval(iouType='segm')`, conf 0.001, NMS 0.7, 100 detecciones, máscaras a resolución original).
Además re-infiere los **3 modelos originales de la semilla 2026** como *control de puente*: si sus AP coinciden con los archivados, el cambio de versión de librerías no altera la comparación.

**Sin entrenamiento. Sin ajuste de umbrales. Sin elegir checkpoints con test.** El test ya fue consultado en experimentos previos: el análisis es **retrospectivo y descriptivo**.

**Cómo usarlo:** Entorno de ejecución → GPU (L4 o T4) → **Ejecutar todas**. No hay que editar nada. Si Colab se desconecta, vuelve a «Ejecutar todas»: continúa donde quedó.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Se necesita GPU: Entorno de ejecución → Cambiar tipo → GPU (L4 o T4).')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%pip -q install "ultralytics==8.4.213" "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "openpyxl>=3.1,<4" "matplotlib>=3.8" pillow

## 1. Drive, verdad de terreno y protocolo congelado (mismas verificaciones que el NB33)

In [ ]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
from contextlib import redirect_stdout
from collections import defaultdict
from IPython.display import display
from PIL import Image
import hashlib, json, io, os, gc, time, platform, shutil, zipfile
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
import torchvision, ultralytics
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils

assert ultralytics.__version__=='8.4.213', f'Versión inesperada de Ultralytics: {ultralytics.__version__}'
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'No se encuentra la carpeta del proyecto: {ROOT}'
SPLIT=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
RES=ROOT/'03_Resultados'; RUN=RES/'06_Entrenamiento'; REPORTS=RES/'05_Reportes'
OUT54=RES/'54_Presupuesto_Equiparado_L4'; OUT55=RES/'55_MaskRCNN_LR_Coseno_L4'
OUT=RES/'56_Evaluacion_Test_Presupuesto_Equiparado'; PRED=OUT/'predicciones'; FIGS=OUT/'figuras'
for d in (OUT,PRED,FIGS): d.mkdir(parents=True,exist_ok=True)
YAML=SPLIT/'data_fecha.yaml'; MANIFEST=SPLIT/'particion_fecha_manifest.json'
BASE09=REPORTS/'CERDOS_09_COCO_Armonizado'; REF09=BASE09/'CERDOS_09_Protocolo.json'; GT_FILE=BASE09/'CERDOS_09_GT_test_coco.json'
BASE23=REPORTS/'CERDOS_23_YOLO26_COCO_Test'; REF23=BASE23/'CERDOS_23_Protocolo.json'
REPORT33=REPORTS/'CERDOS_33_COCO_9_Semillas'/'CERDOS_33_Informe_Final.json'
for f in (YAML,MANIFEST,REF09,GT_FILE,REF23):
    if not f.is_file(): raise FileNotFoundError(f'Falta una dependencia archivada: {f}')

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
    return h.hexdigest()
def readjson(p): return json.loads(Path(p).read_text(encoding='utf-8'))
def write_atomic(p,obj):
    p=Path(p); t=p.with_suffix(p.suffix+'.writing')
    t.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8'); os.replace(t,p)

ref09=readjson(REF09); ref23=readjson(REF23)
assert ref09['pipeline']=='CERDOS_09_COCOv1'
assert int(ref09['max_dets'])==100 and int(ref09['yolo_imgsz'])==640
assert float(ref09['yolo_conf'])==0.001 and float(ref09['yolo_iou_nms'])==0.7 and bool(ref09['yolo_retina_masks']) is True
assert float(ref09['mask_score_thresh'])==0.001 and float(ref09['mask_bin_thresh'])==0.5
cfg=yaml.safe_load(YAML.read_text(encoding='utf-8')); TEST_TXT=Path(cfg['test'])
assert sha256(YAML)==ref09['yaml_sha256'] and sha256(TEST_TXT)==ref09['test_file_sha256']
paths=[Path(l.strip()) for l in TEST_TXT.read_text(encoding='utf-8').splitlines() if l.strip()]
assert len(paths)==373 and len(set(map(str,paths)))==373
manifest=readjson(MANIFEST)['asignaciones_fecha']
assert all(manifest.get(p.name[:10])=='test' for p in paths)
bydate=defaultdict(list)
for iid,p in enumerate(paths,1): bydate[p.name[:10]].append(iid)
assert len(bydate)==5
gt_data=readjson(GT_FILE)
assert len(gt_data['images'])==373 and len(gt_data['annotations'])==4648
for iid,p in enumerate(paths,1):
    im=gt_data['images'][iid-1]; assert int(im['id'])==iid and im['file_name']==p.name
assert ref23['gt_json_sha256']==sha256(GT_FILE)

# Copia local de las 373 imágenes de test (evita la lentitud de Drive). Se verifica tamaño de cada archivo.
CACHE=Path('/content/test_cache'); CACHE.mkdir(exist_ok=True)
LOCAL=[]
for p in paths:
    q=CACHE/p.name
    if not (q.is_file() and q.stat().st_size==p.stat().st_size): shutil.copy2(p,q)
    LOCAL.append(q)
print('Verificado: 373 imágenes / 4.648 máscaras / 5 fechas de test. Copia local lista.')

## 2. Los 15 checkpoints: 12 con presupuesto equiparado + 3 controles originales (semilla 2026)
Cada hash se comprueba contra el `finish.json` del entrenamiento (NB54/NB55) o contra los protocolos archivados (NB09/NB23).

In [ ]:
SEEDS=(2026,2027,2028)
CONFIGS=['YOLO11n-seg','YOLO26n-seg','Mask R-CNN (constant LR)','Mask R-CNN (cosine LR)']
CKPT={}; meta=[]
def add(cfg,seed,family,path,expected_sha,source):
    if not Path(path).is_file(): raise FileNotFoundError(f'Falta el checkpoint: {path}')
    d=sha256(path)
    assert d==expected_sha, f'Hash distinto para {cfg} {seed}: revisar antes de evaluar.'
    CKPT[(cfg,seed)]={'family':family,'path':Path(path),'sha256':d,'source':source}
    meta.append({'Configuration':cfg,'Seed':seed,'Source':source,'Checkpoint':str(path),'SHA256':d})

VAL={}
for seed in SEEDS:
    for cfg,tag,family in [('YOLO11n-seg','yolo11n-seg','yolo'),('YOLO26n-seg','yolo26n-seg','yolo')]:
        folder=OUT54/'matched'/tag/f'seed_{seed}'; fin=readjson(folder/'finish.json')
        assert int(fin['seed'])==seed and fin['test_evaluated'] is False
        add(cfg,seed,family,folder/'weights'/'best.pt',fin['best_weights_sha256'],'NB54'); VAL[(cfg,seed)]=fin['best_validation_seg_AP']
    folder=OUT54/'matched'/'maskrcnn_resnet50_fpn'/f'seed_{seed}'; fin=readjson(folder/'finish.json')
    assert fin['test_evaluated'] is False
    add('Mask R-CNN (constant LR)',seed,'maskrcnn',folder/'maskrcnn_best.pt',fin['best_weights_sha256'],'NB54'); VAL[('Mask R-CNN (constant LR)',seed)]=fin['best_validation_seg_AP']
    folder=OUT55/'maskrcnn_resnet50_fpn_cosine'/f'seed_{seed}'; fin=readjson(folder/'finish.json')
    assert fin['test_evaluated'] is False
    add('Mask R-CNN (cosine LR)',seed,'maskrcnn',folder/'maskrcnn_best.pt',fin['best_weights_sha256'],'NB55'); VAL[('Mask R-CNN (cosine LR)',seed)]=fin['best_validation_seg_AP']

# Controles de puente: modelos originales de la semilla 2026, ya evaluados en NB09/NB23/NB33.
add('CONTROL YOLO11n-seg original',2026,'yolo',RUN/'YOLO11n_fecha_v1'/'weights'/'best.pt',ref09['yolo_sha256'],'NB03/NB09')
add('CONTROL YOLO26n-seg original',2026,'yolo',RUN/'YOLO26n_fecha_v1'/'seed_2026'/'weights'/'best.pt',ref23['weights_sha256']['2026'],'NB22/NB23')
add('CONTROL Mask R-CNN original',2026,'maskrcnn',RUN/'MaskRCNN_fecha_v1'/'maskrcnn_best_model.pt',ref09['mask_sha256'],'NB06/NB09')
assert len(CKPT)==15
display(pd.DataFrame(meta)[['Configuration','Seed','Source']])

## 3. Protocolo congelado del NB56

In [ ]:
PROTO={
 'experiment':'CERDOS_56_Test_Compute_Matched_v1',
 'parent_protocol09_sha256':sha256(REF09),'ground_truth_sha256':sha256(GT_FILE),
 'test_txt_sha256':sha256(TEST_TXT),'yaml_sha256':sha256(YAML),'manifest_sha256':sha256(MANIFEST),
 'checkpoint_sha256':{f'{c} | {s}':v['sha256'] for (c,s),v in CKPT.items()},
 'test_images':373,'test_GT_instances':4648,'test_dates':5,
 'coco_eval':{'iouType':'segm','maxDets':[1,10,100],'IoU thresholds':'0.50:0.05:0.95'},
 'inference':{'YOLO':{'imgsz':640,'conf':0.001,'iou':0.7,'max_det':100,'retina_masks':True,'mask_bin_thresh':0.5},
              'Mask R-CNN':{'min_size':512,'max_size':768,'score_thresh':0.001,'detections_per_img':100,'mask_bin_thresh':0.5}},
 'ultralytics':'8.4.213',
 'selection':'Best checkpoint by validation mask AP (NB54/NB55); test never used for selection or thresholds.',
 'analysis_status':'retrospective, descriptive; test examined in earlier experiments; evaluated once for these checkpoints',
}
PROTO_FILE=OUT/'CERDOS_56_Protocolo.json'
if PROTO_FILE.exists():
    assert readjson(PROTO_FILE)==PROTO, 'El protocolo del NB56 cambió: no mezclar predicciones antiguas.'
else:
    if any(PRED.glob('*.json')): raise RuntimeError('Hay predicciones sin protocolo en la carpeta: revisar.')
    write_atomic(PROTO_FILE,PROTO)
PROTO_HASH=sha256(PROTO_FILE)
print('Protocolo NB56 SHA-256:',PROTO_HASH[:16])

## 4. Inferencia (idéntica al NB33), con guardado cada 25 imágenes y reanudación automática

In [ ]:
def mask_to_rle(mask):
    rle=maskutils.encode(np.asfortranarray(mask.astype(np.uint8))); rle['counts']=rle['counts'].decode('ascii'); return rle

def new_maskrcnn():
    model=maskrcnn_resnet50_fpn(weights=None,weights_backbone=None,min_size=512,max_size=768,trainable_backbone_layers=3)
    model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,2)
    model.roi_heads.mask_predictor=MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,2)
    model.roi_heads.score_thresh=0.001; model.roi_heads.detections_per_img=100
    return model

def validate_rows(rows):
    assert isinstance(rows,list) and len(rows)>0
    for r in rows:
        assert 1<=int(r['image_id'])<=373 and int(r['category_id'])==1 and 0<=float(r['score'])<=1
        assert 'size' in r['segmentation'] and 'counts' in r['segmentation']
    return True

def slug(cfg,seed):
    return (cfg.replace(' ','_').replace('(','').replace(')','').replace('-','')+f'_seed_{seed}')

def predict(cfg,seed):
    info=CKPT[(cfg,seed)]; stem='CERDOS_56_Pred_'+slug(cfg,seed)
    path=PRED/(stem+'.json'); lock=PRED/(stem+'_hash.json'); partial=PRED/(stem+'_partial.json')
    if path.exists():
        m=readjson(lock); assert m['protocol_sha256']==PROTO_HASH and m['weights_sha256']==info['sha256'] and m['predictions_sha256']==sha256(path)
        print('Reutilizando',stem); return path
    if partial.exists():
        b=readjson(partial); assert b['protocol_sha256']==PROTO_HASH and b['weights_sha256']==info['sha256']
        done=int(b['last_image_id']); rows=b['predictions']; print(f'Reanudando {stem} desde la imagen {done}/373')
    else: done=0; rows=[]
    if info['family']=='yolo':
        model=YOLO(str(info['path'])); assert model.task=='segment'
    else:
        model=new_maskrcnn().to('cuda:0')
        ck=torch.load(info['path'],map_location='cpu',weights_only=False)
        model.load_state_dict(ck['model'],strict=True); del ck; model.eval()
    t0=time.perf_counter()
    for iid in range(done+1,374):
        p=LOCAL[iid-1]; im=gt_data['images'][iid-1]; ow,oh=int(im['width']),int(im['height'])
        if info['family']=='yolo':
            r=model.predict(source=str(p),device=0,imgsz=640,conf=0.001,iou=0.7,max_det=100,retina_masks=True,verbose=False,save=False)[0]
            scores=r.boxes.conf.detach().cpu().numpy() if r.boxes is not None else np.empty(0)
            masks=r.masks.data.detach().cpu().numpy() if len(scores) else np.zeros((0,oh,ow))
        else:
            with Image.open(p) as image: rgb=np.asarray(image.convert('RGB')).copy()
            assert rgb.shape[:2]==(oh,ow)
            with torch.inference_mode(): out=model([to_tensor(rgb).to('cuda:0')])[0]
            scores=out['scores'].detach().cpu().numpy()
            masks=out['masks'].detach().cpu().numpy()[:,0] if len(scores) else np.zeros((0,oh,ow))
        assert masks.shape==(len(scores),oh,ow),(iid,masks.shape)
        for s,m in zip(scores,masks):
            b=(m>0.5).astype(np.uint8)
            if b.any(): rows.append({'image_id':iid,'category_id':1,'segmentation':mask_to_rle(b),'score':float(s)})
        if iid%25==0 or iid==373:
            write_atomic(partial,{'protocol_sha256':PROTO_HASH,'weights_sha256':info['sha256'],'last_image_id':iid,'predictions':rows})
    validate_rows(rows); write_atomic(path,rows)
    write_atomic(lock,{'protocol_sha256':PROTO_HASH,'weights_sha256':info['sha256'],'predictions_sha256':sha256(path),
                       'n_predictions':len(rows),'finished_utc':datetime.now(timezone.utc).isoformat(),
                       'seconds_this_session':round(time.perf_counter()-t0,1)})
    if partial.exists(): partial.unlink()
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f'Listo {stem}: {len(rows)} máscaras en {time.perf_counter()-t0:.0f} s'); return path

PREDS={}
for k in CKPT: PREDS[k]=predict(*k)
print('15 conjuntos de predicciones disponibles.')

## 5. COCO mask AP (global y por fecha) + control de puente

In [ ]:
gt=COCO(); gt.dataset=gt_data
with redirect_stdout(io.StringIO()): gt.createIndex()
def coco_ap(rows,imgids=None):
    safe=[{'image_id':int(r['image_id']),'category_id':1,'segmentation':r['segmentation'],'score':float(r['score'])} for r in rows]
    with redirect_stdout(io.StringIO()): dt=gt.loadRes(safe)
    ev=COCOeval(gt,dt,'segm'); ev.params.maxDets=[1,10,100]
    if imgids is not None: ev.params.imgIds=sorted(map(int,imgids))
    with redirect_stdout(io.StringIO()): ev.evaluate(); ev.accumulate(); ev.summarize()
    return {'Mask_AP50_95':float(ev.stats[0]),'Mask_AP50':float(ev.stats[1]),'Mask_AP75':float(ev.stats[2])}

records=[]; per_date=[]
for (cfg,seed),path in PREDS.items():
    rows=readjson(path); validate_rows(rows); g=coco_ap(rows)
    records.append({'Configuration':cfg,'Seed':seed,'Predicted_instances':len(rows),'Best_val_AP':VAL.get((cfg,seed)),**g})
    for date,ids in sorted(bydate.items()):
        per_date.append({'Configuration':cfg,'Seed':seed,'Date':date,'Images':len(ids),**coco_ap(rows,ids)})
    print(f'{cfg:30s} {seed}  test mask AP50:95 = {g["Mask_AP50_95"]:.4f}')
df_all=pd.DataFrame(records); df_date_all=pd.DataFrame(per_date)

# Valores originales archivados (NB33). Si el informe no estuviera, se usan los publicados en el manuscrito.
ORIG={('YOLO11n-seg',2026):0.8655,('YOLO11n-seg',2027):0.8610,('YOLO11n-seg',2028):0.8635,
      ('YOLO26n-seg',2026):0.8495,('YOLO26n-seg',2027):0.8500,('YOLO26n-seg',2028):0.8425,
      ('Mask R-CNN',2026):0.7962,('Mask R-CNN',2027):0.7976,('Mask R-CNN',2028):0.7904}
if REPORT33.is_file():
    for r in readjson(REPORT33)['global']: ORIG[(r['Model'],int(r['Seed']))]=float(r['Mask_AP50_95'])
    print('Valores originales tomados del informe NB33.')
else: print('Aviso: no se encontró el informe NB33; se usan los valores publicados (4 decimales).')

bridge=[]
for cfg,orig in [('CONTROL YOLO11n-seg original','YOLO11n-seg'),('CONTROL YOLO26n-seg original','YOLO26n-seg'),('CONTROL Mask R-CNN original','Mask R-CNN')]:
    new=float(df_all[(df_all.Configuration==cfg)].Mask_AP50_95.iloc[0]); old=ORIG[(orig,2026)]
    bridge.append({'Model':orig+' (seed 2026)','Archived_AP_NB33':old,'Re-inferred_AP_NB56':new,'Difference':new-old,
                   'Status':'OK' if abs(new-old)<=0.002 else 'REVISAR: diferencia > 0.002'})
df_bridge=pd.DataFrame(bridge); display(df_bridge.round(5))

## 6. Resumen: presupuesto equiparado frente a recetas originales (media ± DE de 3 semillas)

In [ ]:
df_global=df_all[df_all.Configuration.isin(CONFIGS)].copy()
df_date=df_date_all[df_date_all.Configuration.isin(CONFIGS)].copy()
assert len(df_global)==12 and len(df_date)==60
summary=[]
for cfg in CONFIGS:
    f=df_global[df_global.Configuration==cfg].sort_values('Seed')
    d={'Configuration':cfg}
    for m in ('Mask_AP50_95','Mask_AP50','Mask_AP75'):
        v=f[m].to_numpy(float); d[m+'_mean']=v.mean(); d[m+'_sd']=v.std(ddof=1)
    d['AP50_95_seeds']=' / '.join(f'{x:.4f}' for x in f.Mask_AP50_95)
    base='Mask R-CNN' if cfg.startswith('Mask') else cfg
    o=np.array([ORIG[(base,s)] for s in SEEDS]); d['Original_recipe_mean']=o.mean()
    d['Change_vs_original_points']=100*(d['Mask_AP50_95_mean']-o.mean())
    summary.append(d)
df_summary=pd.DataFrame(summary)
m=df_summary.set_index('Configuration')['Mask_AP50_95_mean']
diffs=[]
for a,b in [('YOLO11n-seg','Mask R-CNN (constant LR)'),('YOLO11n-seg','Mask R-CNN (cosine LR)'),
            ('YOLO26n-seg','Mask R-CNN (cosine LR)'),('YOLO11n-seg','YOLO26n-seg'),('Mask R-CNN (cosine LR)','Mask R-CNN (constant LR)')]:
    per=[100*(float(df_global[(df_global.Configuration==a)&(df_global.Seed==s)].Mask_AP50_95.iloc[0])-
              float(df_global[(df_global.Configuration==b)&(df_global.Seed==s)].Mask_AP50_95.iloc[0])) for s in SEEDS]
    diffs.append({'Comparison':f'{a} minus {b}','Mean_difference_points':100*(m[a]-m[b]),
                  'Per_seed_points':' / '.join(f'{x:+.2f}' for x in per),'Same_sign_all_seeds':bool(all(x>0 for x in per) or all(x<0 for x in per))})
df_diffs=pd.DataFrame(diffs)
# Orden por fecha (media de semillas) para cada configuración
mean_date=df_date.groupby(['Configuration','Date'])['Mask_AP50_95'].mean().unstack('Date').loc[CONFIGS]
order_by_date=pd.DataFrame({d:[' > '.join(mean_date[d].sort_values(ascending=False).index)] for d in mean_date.columns}).T.rename(columns={0:'Order'})
pd.set_option('display.max_colwidth',120)
print('TEST (presupuesto equiparado, 60 min L4):')
display(df_summary[['Configuration','AP50_95_seeds','Mask_AP50_95_mean','Mask_AP50_95_sd','Original_recipe_mean','Change_vs_original_points']].round(4))
display(df_diffs.round(2)); display(mean_date.round(4)); display(order_by_date)

## 7. Figuras para el artículo (color, sin título dentro de la imagen; PNG 600 dpi, PDF y EPS)

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.labelsize':9,'xtick.labelsize':8,'ytick.labelsize':8})
COL={'YOLO11n-seg':'#1F5A8C','YOLO26n-seg':'#2E8B6E','Mask R-CNN (constant LR)':'#B5652E','Mask R-CNN (cosine LR)':'#7A3E9D'}
MARK={'YOLO11n-seg':'o','YOLO26n-seg':'s','Mask R-CNN (constant LR)':'^','Mask R-CNN (cosine LR)':'D'}
LBL={'YOLO11n-seg':'YOLO11n-seg','YOLO26n-seg':'YOLO26n-seg','Mask R-CNN (constant LR)':'Mask R-CNN\nconstant LR','Mask R-CNN (cosine LR)':'Mask R-CNN\ncosine LR'}
fig,ax=plt.subplots(figsize=(6.85,3.0))
for i,cfg in enumerate(CONFIGS):
    f=df_global[df_global.Configuration==cfg].sort_values('Seed')
    x=np.array([i-.12,i,i+.12])
    ax.scatter(x,f.Mask_AP50_95,s=34,marker=MARK[cfg],color=COL[cfg],edgecolor='black',linewidth=.4,zorder=3,label='Equal 60-min budget' if i==0 else None)
    mn=f.Mask_AP50_95.mean(); ax.plot([i-.25,i+.25],[mn,mn],color='black',lw=1.2,zorder=4)
    base='Mask R-CNN' if cfg.startswith('Mask') else cfg
    o=np.mean([ORIG[(base,s)] for s in SEEDS]); ax.plot([i-.25,i+.25],[o,o],color='grey',lw=1.2,ls='--',zorder=2,label='Original recipe (mean)' if i==0 else None)
ax.set_xticks(range(4),[LBL[c] for c in CONFIGS]); ax.set_ylabel('Test mask AP@[0.50:0.95]')
ax.grid(axis='y',alpha=.25); ax.legend(frameon=False,loc='lower left',fontsize=8)
fig.tight_layout()
for ext in ('png','pdf','eps'): fig.savefig(FIGS/f'Fig_test_AP_compute_matched.{ext}',dpi=600,bbox_inches='tight')
plt.show(); plt.close(fig)

dates=list(mean_date.columns)
fig,ax=plt.subplots(figsize=(6.85,2.6))
for cfg in CONFIGS:
    ax.plot(range(len(dates)),mean_date.loc[cfg].to_numpy(),marker=MARK[cfg],color=COL[cfg],lw=1.3,ms=5,label=LBL[cfg].replace('\n',' '))
ax.set_xticks(range(len(dates)),dates); ax.set_xlabel('Test acquisition date'); ax.set_ylabel('Mask AP@[0.50:0.95]\n(mean of 3 seeds)')
ax.grid(alpha=.25); ax.legend(frameon=False,ncol=2,fontsize=7.5,loc='lower left')
fig.tight_layout()
for ext in ('png','pdf','eps'): fig.savefig(FIGS/f'Fig_test_AP_by_date_compute_matched.{ext}',dpi=600,bbox_inches='tight')
plt.show(); plt.close(fig)

## 8. Excel, informe JSON y ZIP de resultados

In [ ]:
EXCEL=OUT/'CERDOS_56_Test_Presupuesto_Equiparado.xlsx'; REPORT_FILE=OUT/'CERDOS_56_Informe_Final.json'
def recs(df): return json.loads(df.to_json(orient='records'))
REPORT={'experiment':56,'created_utc':datetime.now(timezone.utc).isoformat(),'protocol':PROTO,'protocol_sha256':PROTO_HASH,
 'software':{'python':platform.python_version(),'torch':torch.__version__,'torchvision':torchvision.__version__,
             'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(0)},
 'global':recs(df_all),'per_date':recs(df_date_all),'summary':recs(df_summary),
 'differences':recs(df_diffs),'bridge_check':recs(df_bridge),
 'prediction_sha256':{f'{c} | {s}':sha256(p) for (c,s),p in PREDS.items()},
 'test_is_new_independent_validation':False}
with pd.ExcelWriter(EXCEL,engine='openpyxl') as w:
    for name,df in [('Global_15',df_all),('Date',df_date_all),('Summary',df_summary),('Differences',df_diffs),
                    ('Bridge_check',df_bridge),('Checkpoints',pd.DataFrame(meta))]:
        df.to_excel(w,sheet_name=name,index=False); w.sheets[name].freeze_panes='A2'
write_atomic(REPORT_FILE,REPORT)
ZIP=OUT/'CERDOS_56_Resultados.zip'
OUTFILES=[EXCEL,REPORT_FILE,PROTO_FILE]+sorted(FIGS.glob('*'))+sorted(PRED.glob('*_hash.json'))
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTFILES: z.write(p,arcname=p.name)
print('Excel:',EXCEL); print('Informe:',REPORT_FILE); print('ZIP:',ZIP)
print('\n✅ NB56 TERMINADO. Avisar para revisar los resultados.')
try: files.download(str(ZIP))
except Exception as e: print('Descarga automática no disponible; el ZIP está en Drive.')

## Interpretación (para el manuscrito)
- Estos AP de test son la **cifra final** de los modelos con presupuesto equiparado; se comparan con los 9 originales del NB33 bajo el mismo evaluador.
- El **control de puente** debe mostrar diferencias ≤ 0,002 para afirmar que el cambio de versión de Ultralytics/torch no altera la comparación.
- El test se había consultado antes: el resultado es **retrospectivo y descriptivo**, no una validación ciega ni externa.